In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

#### api 키
- YOUR_OPENAI_API_KEY

- 의존성 라이브러리 설치

In [1]:
# LangChain 기본 구조 및 핵심 모듈
%pip install -qU langchain langchain-core langchain-community langchain-openai

# langchain-chroma, faiss-cpu : 벡터 DB를 오픈소스로 제공
%pip install -q langchain-chroma faiss-cpu

# langchain-teddynote   : 테디노트(유튜버)가 만든 랭체인 확장 패키지
# !pip install -qU langchain-teddynote
# 25.11.25 수업 중 다수가 설치 안 됨 → 서버측 에러인듯?

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


- API 키 등록

In [4]:
import os

api_key = "YOUR_OPENAI_API_KEY"
os.environ["OPENAI_API_KEY"] = api_key

### `LC01_LangChain 기초, LLM 연동....ipynb`에서 수행한 건 **<U>Zero-Shot Prompting**</U>
- MessagePromptTemplate (역할부여) 제외


## 제로샷 프롬프팅 (Zero-Shot Prompting)
- Zero-Shot : AI 모델이 사전 학습 없이 새로운 작업이나 상황에 대응하는 방식
- Zero-Shot Prompting : **사전 예시 없이**, 단순히 요청만 주어져 모델이 답변을 생성하는 방식

> **특징**  
> - **<font color=red>추가 예시 없이</font>** 문제 설명만으로 작업을 수행함  
> - 모델은 오로지 프롬프트에 포함된 정보만을 기반으로 응답을 생성함  
> - **장점** : `빠르고 간단하게 적용할 수 있음`  
> - **단점** : `복잡한 문제나 구체적인 형식을 요구하는 작업에서는 모델의 응답이 기대에 못 미칠 수 있음`

### Zero-Shot Prompting 개선 방법
- Few-Shot Prompting
- 메모리 기능 사용
- 체인 결합 (여러개의 언어 모델 연결)
- RAG (검색, 증강, 생성)
- TAG (테이블(DB), 증강, 생성)
- Agent 등



## 퓨샷 프롬프팅 (Few-Shot Prompting)
- 몇 가지 예시(사례)를 프롬프트에 포함시켜, 모델이 원하는 출력 형식이나 문제 해결 방식을 학습하도록 돕는 방식

> **특징**  
> - <b><font color=red>예시를 제공</font></b>하여 모델에게 어떤 식으로 답변해야 하는지 안내함  
> - 예시를 통해 맥락을 명확하게 전달하고, 응답의 일관성과 정확도를 향상시킴  
> - **장점** : `모델이 복잡하거나 구체적인 작업을 보다 정확하게 수행할 수 있음`  
> - **단점** : `적절한 예시 선택이 중요하며, 프롬프트가 길어질 수 있음`

<br>

---

<br>

- 단순한 지시만으로는 모델이 의도된 작업을 충분히 이해하기 어려운 복잡하거나 미묘한 작업에 유용
- 모델 Finetuning이나 대규모 데이터셋이 실행 가능하지 않은 상황에서 LLM의 능력을 유연하고 효율적으로 활용하기 위한 강력한 도구

In [7]:
from langchain.chat_models import init_chat_model # 모델 초기화 클래스
from langchain_core.output_parsers import StrOutputParser

# 모델에 전달해 줄 프롬포트를 정의하는 도구
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import FewShotPromptTemplate

In [9]:
# (예시 1) 간단한 "반대말 출력하기" 데이터셋
examples1 = [
    {"input": "happy","output": "sad"},
    {"input": "tall","output": "short"},
    {"input": "sunny","output": "rainy"},
    {"input": "surprised","output": "calm"},
    {"input": "dry","output": "humid"},
    {"input": "hot","output": "cold"},
    {"input": "satisfied","output": "unsatisfied"}
]

In [10]:
# 모델 초기화
llm = init_chat_model(model="openai:gpt-4.1", # 사용할 모델 설정
                      temperature=0,          # 0~2 창의성과 안정성 정해주는 변수
                      max_tokens=500          # 출력 최대 길이
                      )

In [11]:
# PromptTemplate: 예시의 "형식(포맷, 템플릿)" 정의 → Input, Output을 어떤 형태로 LLM에게 보낼지
prompt = PromptTemplate.from_template(
    "Input: {input}\nOutput: {output}"
)
# 각 예시가 아래의 문자열 형태로
# Input: happy
# Output: sad

In [13]:
# FewShotPromptTemplate 생성

# 예시(example) + 예시 형식(example_prompt) + 지시문(prefix/suffix)을 조합한 최종 프롬프트
fewshot_prompt = FewShotPromptTemplate(
    examples = examples1,                 # LLM에게 보여줄 input-output 예시
    example_prompt = prompt,              # 각 예시가 어떤 형태로 출력될지
    prefix = "입력의 반대말을 출력한다.", # 예시 앞부분에 위치 항상 붙는 문장(모델에 작업 규칙 설명)
    suffix = "input: {input}",            # 예시 뒤에서, 실제 사용자의 입력이 들어가는 실제 질문 부분
    input_variables = ["input"]
)

In [14]:
print(fewshot_prompt.format(input="여름"))

입력의 반대말을 출력한다.

Input: happy
Output: sad

Input: tall
Output: short

Input: sunny
Output: rainy

Input: surprised
Output: calm

Input: dry
Output: humid

Input: hot
Output: cold

Input: satisfied
Output: unsatisfied

input: 여름


[ 실습 ]

In [ ]:
from langchain_teddynote.messages import stream_response

In [ ]:
examples2 = [
    {
        "question":"철수는 빨강색을 좋아하고 영희는 파란색을 좋아한다.",
        "answer":'"철수":"빨강색", "영희":"파란색"'
    },
    {
        "question":"길동은 짜장을 좋아하고, 심청은 짬뽕을 좋아한다.",
        "answer":'"길동":"짜장","심청":"짬뽕"'
    },
    {
        "question":"맹구는 국어를 좋아하고 유리는 수학을 좋아한다.",
        "answer":'"맹구":"국어", "유리":"수학"'
    },
    {
        "question":"봉봉은 AI를 강의하고 균균은 사물인터넷을 강의한다.",
        "answer":'"봉봉":"AI","균균":"사물인터넷"'
    },
    {
        "question":"A는 서울에 살고, B는 제주도에 산다.",
        "answer":'"A":"서울", "B":"제주도"'
    }
]

In [ ]:
prompt = PromptTemplate.from_template("입력: {question}\n 출력: {answer}")

# fewshot prompt template
fewshot_prompt = FewShotPromptTemplate(
    example = examples2,
    example_prompt = prompt,
    prefix = "핵심 키워드만 정리한다.",
    suffix = ""
)

# Callbacks - 스트리밍(streaming)

- <font color=red>stream_response()</font> : 스트림 형태로 결과 출력
  - **<U>결괏값을 반환하지 않음**</U>
  - import하여 새롭게 초기화 하지 않는 경우, 2번씩 출력되는 문제 발생 `가끔 출력이 2번씩 나오는 문제가 존재함` (버전 업데이트 이전)
  - 다른 Callback 활용 가능

- 스트리밍 옵션은 <font color=red>질의에 대한 답변을 실시간</font>으로 받을 때 유용
- <font color=red>streaming=True</font>로 설정하고 스트리밍으로 답변을 받기 위한 <font color=red>StreamingStdOutCallbackHandler()</font>을 콜백으로 지정

> 오류가 뜨는 경우 : 라이브러리가 langchain.callbacks.streaming_stdout에서 langchain_core.callbacks.streaming_stdout로 변경됨

[ 실습 ]

In [ ]:
# Few‐Shot 예시: LangChain 주요 구성요소 설명
example3 = [
    {"input" : "Agent",           "output" : "프롬프트를 통해 도구를 호출하며 워크플로우를 제어할 수 있는 자동화 컴포넌트"},
    {"input" : "Chain",           "output" : "여러 프롬프트와 모델 호출을 순차적으로 연결해 파이프라인을 구성하는 구조"},
    {"input" : "PromptTemplate",  "output" : "입력 변수를 포함해 모델에 보낼 프롬프트 형식을 정의하는 클래스"}
]

# ExampleSelector
- 예시가 많으면 토큰 사용량이 많아지고 따라서 API 사용료가 많이 나옴
- 최종 질문과 유사한 예시 한 두 개만 프롬프트에 포함할 예제를 선택하는 경우에 사용

- 종류
  - <font color=red>SemanticSimilarityExampleSelector</font>: 벡터 임베딩의 코사인 유사도를 이용해 질문과 의미가 가까운 예시를 선택
---
- 상황에 따라 사용
  - <font color=red>BaseExampleSelector</font>: 상속 받아 사용자 정의 ExampleSelector를 생성
  - <font color=red>LengthBasedExampleSelector</font>: 지정 길이가 넘어가지 않도록 예시 개수를 조절
  - <font color=red>MaxMarginalRelevanceExampleSelector</font>: Maximal Marginal Relevance (MMR)을 이용해 질문과 가까우면서도 다양한 예시를 선택
  - <font color=red>NGramOverlapExampleSelector</font>: n-gram overlap score를 이용해 질문과 가까운 예시를 선택

- 사용자 입력과의 유사도에 따라 예시를 선택해서 LLM 모델에 Few-show Prompting 하기 (질문에 대해 예시와 같이 답변하도록 하는 기능 구현)
  - 예시와 사용자 입력을 Text Embeddings 변환하여, Cosine Similarity가 가장 높은 k개의 예시를 선별해서 Few-shot Prompt에 넣어서 LLM 모델에 전달하는 방법

### 📌 Chroma vs FAISS 비교표

| 구분               | **Chroma**                 | **FAISS**                               |
| ---------------- | -------------------------- | --------------------------------------- |
| **설명**           | 로컬·파일 기반으로 쉽게 쓰는 가벼운 벡터 DB | 메타(Facebook)가 개발한 고성능 벡터 검색 엔진          |
| **설치/사용 난이도**    | ⭐ 매우 쉬움                    | ⭐⭐ 중간(조금 더 기술적)                         |
| **저장 방식**        | SQLite/Local 파일 기반         | 메모리/인덱스 파일 기반                           |
| **검색 속도**        | 중간                         | 매우 빠름(고성능)                              |
| **대규모 데이터**      | 중·소규모 데이터에 적합              | 대량(수십~수백만 벡터)에서도 고속 검색 가능               |
| **데이터 지속성**      | 파일로 저장 → 재사용 가능            | 인메모리 중심, 직접 저장을 구현해야 함(저장/로드 지원은 있음)    |
| **필요한 구성 요소**    | 파이썬만 있으면 동작(가볍고 간단)        | CPU 또는 GPU 환경 필요(특히 GPU 버전 매우 빠름)       |
| **문서화 & 예제**     | 교육용·입문용으로 설명이 쉽고 예제가 많음    | ML/IR(Information Retrieval) 영역에서 많이 쓰임 |
| **LangChain 연동** | `langchain-chroma`로 간단     | `FAISS.from_documents()`로 쉽게 연동 가능      |
| **장점**           | - 설치·사용이 매우 간단             |                                         |


In [ ]:
# 1) 긴 회의록 문장을 주면 → 정리된 회의록 형식으로 출력
# 2) 보고서 설명을 주면 → 핵심 요약을 출력

examples4 = [
    {
        "input": (
            "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. "
            "회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. "
            "회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. "
            "팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다."
        ),
        "answer": (
            "회의록: XYZ 회사 마케팅 전략 회의\n"
            "일시: 2023년 12월 25일\n"
            "장소: XYZ 회사 회의실\n"
            "참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)\n\n"
            "1. 개회\n"
            "   - 김수진 팀장의 개회사\n"
            "2. 시장 동향 개요 (김수진)\n"
            "   - 최근 시장 동향 분석\n"
            "3. 디지털 마케팅 전략 (박지민)\n"
            "   - SEO 최적화 및 온라인 광고 방안\n"
            "4. 소셜 미디어 캠페인 (이준호)\n"
            "   - 인플루언서 마케팅 제안\n"
            "5. 종합 논의\n"
            "   - 예산 및 자원 배분\n"
            "6. 마무리\n"
            "   - 다음 회의 일정 및 회의록 배포 담당자 지정"
        )
    },
    {
        "input": (
            "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. "
            "보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 지속 가능한 개발 전략을 다루고 있습니다. "
            "여러 국가의 성공 사례와 얻은 교훈도 포함되어 있습니다."
        ),
        "answer": (
            "문서 요약: 지속 가능한 도시 개발 전략 보고서\n\n"
            "- 중요성: 사회적·경제적·환경적 이점 강조\n"
            "- 문제점: 환경 오염, 자원 고갈, 불평등 분석\n"
            "- 전략: 친환경 건축, 대중교통 개선, 에너지 효율성 증대\n"
            "- 사례 연구: 코펜하겐, 요코하마 성공 사례\n"
            "- 교훈: 다각적 접근, 지역사회 협력, 장기 계획 필요"
        )
    }
]

# MessagePromptTemplate 활용 - 역할을 부여하는 것
- <font color=red>AI 메시지, 시스템 메시지, 사용자 메시지를 사용</font>하여 적절한 맥락을 유지하고, 사용자와의 상호작용을 보다 자연스럽게 처리
  - <font color=red>SystemMessagePromptTemplate</font> : 시스템 프롬프트 설정
  - <font color=red>HumanMessagePromptTemplate</font> : 사용자 프롬프트 설정
- <font color=red>ChatPromptTemplate.from_messages()</font> : 시스템 메시지와 사용자 메시지 템플릿을 포함하는 챗 프롬프트를 구성
- <font color=red>chat_prompt.format_messages()</font> : 사용자의 질문을 포함한 메시지 리스트를 동적으로 생성하여 반환

- 생성된 메시지 리스트는 대화형 인터페이스나 언어 모델과의 상호작용을 위한 입력으로 사용
- 각 메시지는 <font color=red>role</font> (메시지를 말하는 주체, 여기서는 system 또는 user)과 <font color=red>content</font> (메시지의 내용) 속성을 포함
- 이 구조는 시스템과 사용자 간의 대화 흐름을 명확하게 표현하며, 언어 모델이 이를 기반으로 적절한 응답을 생성할 수 있도록 지원

- 또 다른 사용법

```python
# 시스템 메시지 정의
system_message = SystemMessagePromptTemplate.from_template("이 시스템은 천문학 질문에 답변할 수 있습니다")

# 인간 사용자 메시지 정의
human_message = HumanMessagePromptTemplate.from_template("{user_input}")

# 대화형 템플릿 생성
chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message])
```

# FewShotChatMessagePromptTemplate
- <font color=red>Example Selector를 설정해주고, ChatPromptTemplate.from_messages()에 "human"과 "ai"의 메시지 포맷을 설정</font>
- 입력으로 받는 변수 input_variables=["input"] 을 설정

- Example Selector 의 유사도 검색 문제 해결
  - 유사도 계산시 instruction 과 input 을 사용하고 있지만, instruction 만 사용하여 검색시 제대로된 유사도 결과가 나오지 않음
  - 이를 해결하기 위해 커스텀 유사도 계산을 위한 클래스를 정의하여 사용

In [ ]:
examples5 = [
    {
        "ex_instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "ex_input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",
        "ex_answer": """
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
"""
    },
    {
        "ex_instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "ex_input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "ex_answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
"""
    }
]

# 여러 개의 프롬프트 연결하기
- PromptTemplate을 이용해서 <font color=red>+</font> 연산자를 이용해서 여러개의 템플릿을 연결
  - 문자열 + 문자열
  - PromptTemplate + PromptTemplate
  - PromptTemplate + 문자열

---
- 프롬프트 조립용

- [실습] 프롬프트 추가해보기

# 순차적인 체인 연결
- RunableParellel() 은 병렬 수행
- 첫 번째 체인 : 한국어 단어를 영어로 번역하는 작업을 수행
- 두 번째 체인 : 해당 단어를 설명하는 작업을 수행